# BorgRW File Processing Demo

This notebook demonstrates two common workflows for nondominated solution files:
1. Convert a **superheader CSV** into a normal single-header CSV.
2. Split files into **decisions-only** and **objectives-only** tables for downstream plotting (for example, HiPlot).

## Before You Run
This notebook assumes the BorgRWProblems repository is available locally and the package is installed in the active Python environment (editable install is fine).
It will read sample CSVs from `example_scripts/file_processing_demo/data` and write outputs into `example_scripts/file_processing_demo/outputs`.

In [1]:
from pathlib import Path
import csv

from borgRWhelper.file_processing import (
    has_superheader,
    load_solutions_dataframe,
    convert_solutions_csv_to_single_header,
    split_solutions_csv,
)
from borgRWhelper.tradeoff import parallel_plot_hp

## Setup: Paths
The cell below locates the repository root by walking up from the notebook's working directory until it finds `pyproject.toml`. All input and output paths are then derived relative to that root, so the notebook works regardless of where VS Code opened it from.

In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root (pyproject.toml not found).')

cwd = Path.cwd().resolve()
repo_root = find_repo_root(cwd)
demo_dir = repo_root / 'example_scripts' / 'file_processing_demo'
data_dir = demo_dir / 'data'
output_dir = demo_dir / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

superheader_csv = data_dir / 'NondominatedSolutions.superheader.csv'
single_header_csv = data_dir / 'TradeoffAnalysis.single_header.csv'

print('Repo root:      ', repo_root)
print('Demo directory: ', demo_dir)
print('Superheader CSV:', superheader_csv.name)
print('Single-header CSV:', single_header_csv.name)

Repo root:       C:\GitHub\BorgRWProblems
Demo directory:  C:\GitHub\BorgRWProblems\example_scripts\file_processing_demo
Superheader CSV: NondominatedSolutions.superheader.csv
Single-header CSV: TradeoffAnalysis.single_header.csv


## Quick Header Inspection
Inspect the first two lines and run automatic superheader detection.

In [3]:
def preview_first_rows(path: Path, n: int = 2):
    with open(path, 'r', encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f)
        rows = []
        for _ in range(n):
            try:
                rows.append(next(reader))
            except StopIteration:
                break
    return rows

for path in [superheader_csv, single_header_csv]:
    print(f'\n{path.name}')
    print('  has_superheader:', has_superheader(path))
    rows = preview_first_rows(path, n=2)
    for i, row in enumerate(rows, start=1):
        print(f'  row {i}:', row[:8], '...' if len(row) > 8 else '')


NondominatedSolutions.superheader.csv
  has_superheader: True
  row 1: ['', 'Decision Variables', '', '', '', '', 'Objectives', ''] ...
  row 2: ['Solution', 'Cedar Div Min Elev', 'Dogwood Div Min Elev', 'Hickory Elev Min to Irrig', 'Cedar Elev Limits Row 0', 'Cedar Elev Limits Row 1', 'Energy (Negated)', 'Spill'] ...

TradeoffAnalysis.single_header.csv
  has_superheader: False
  row 1: ['Solution', 'Hayden Flood Elev', 'Carson Flood Elev', 'Carson Cons Elev', 'Max Channel Flow', 'Irrig Allocation', 'Hayden1870', 'CityFlow1500'] ...
  row 2: ['12', '1864', '2160.25', '2160', '1500', '1', '0.1', '0.03'] ...


## 1) Convert Superheader to Single Header

In [4]:
converted_csv = output_dir / 'NondominatedSolutions.single_header.csv'
written_path = convert_solutions_csv_to_single_header(
    superheader_csv,
    output_csv=converted_csv,
    superheader='auto',
)

converted_df = load_solutions_dataframe(written_path, superheader='no')
print('Wrote:', written_path)
print('Shape:', converted_df.shape)
print('Columns:', list(converted_df.columns))

Wrote: C:\GitHub\BorgRWProblems\example_scripts\file_processing_demo\outputs\NondominatedSolutions.single_header.csv
Shape: (76, 12)
Columns: ['Solution', 'Cedar Div Min Elev', 'Dogwood Div Min Elev', 'Hickory Elev Min to Irrig', 'Cedar Elev Limits Row 0', 'Cedar Elev Limits Row 1', 'Energy (Negated)', 'Spill', 'West Basin Irrig (Negated)', 'East Basin Irrig (Negated)', 'Total Depletion Shortage', 'Transbasin Diversion']


## 2) Split Into Decisions and Objectives

### A. Superheader file (automatic grouping)

In [5]:
super_split = split_solutions_csv(
    superheader_csv,
    decisions_csv=output_dir / 'NondominatedSolutions.decisions.csv',
    objectives_csv=output_dir / 'NondominatedSolutions.objectives.csv',
    superheader='auto',
)

print('Decision columns:')
print(super_split['decision_columns'])
print('\nObjective columns:')
print(super_split['objective_columns'])
print('\nWrote:')
print('  ', super_split['decisions_path'])
print('  ', super_split['objectives_path'])

Decision columns:
['Cedar Div Min Elev', 'Dogwood Div Min Elev', 'Hickory Elev Min to Irrig', 'Cedar Elev Limits Row 0', 'Cedar Elev Limits Row 1']

Objective columns:
['Energy (Negated)', 'Spill', 'West Basin Irrig (Negated)', 'East Basin Irrig (Negated)', 'Total Depletion Shortage']

Wrote:
   C:\GitHub\BorgRWProblems\example_scripts\file_processing_demo\outputs\NondominatedSolutions.decisions.csv
   C:\GitHub\BorgRWProblems\example_scripts\file_processing_demo\outputs\NondominatedSolutions.objectives.csv


### B. Single-header file (explicit grouping)

For files without superheaders, provide decision/objective columns explicitly.

In [ ]:
decision_cols = [
    'Hayden Flood Elev',
    'Carson Flood Elev',
    'Carson Cons Elev',
    'Max Channel Flow',
    'Irrig Allocation',
]
objective_cols = [
    'Hayden1870',
    'CityFlow1500',
    'TimeCarsonSpill',
    'MaxHaydenPE',
    'AppleValleyVolRelNeg',
    'BorderVolRelNeg',
]

single_split = split_solutions_csv(
    single_header_csv,
    decisions_csv=output_dir / 'TradeoffAnalysis.decisions.csv',
    objectives_csv=output_dir / 'TradeoffAnalysis.objectives.csv',
    superheader='no',
    decision_columns=decision_cols,
    objective_columns=objective_cols,
)

print('Wrote:')
print('  ', single_split['decisions_path'])
print('  ', single_split['objectives_path'])

## 3) Optional: Create an Objectives-Only HiPlot HTML

In [ ]:
obj_df = load_solutions_dataframe(output_dir / 'TradeoffAnalysis.objectives.csv', superheader='no')
hiplot_html = output_dir / 'TradeoffAnalysis.objectives.hiplot.html'

parallel_plot_hp(
    obj_df,
    color_column='BorderVolRelNeg',
    hide_columns=['Solution'],
).to_html(hiplot_html)

print('Wrote:', hiplot_html)